In [1]:
import pandas as pd
import numpy as np

In [2]:
producteurs = pd.read_csv('producteurs.csv')
ventes = pd.read_csv("ventes.csv")
recoltes = pd.read_csv("recoltes.csv")

# EXPLORATION

In [3]:
print(f"La taille du fichier producteurs est : {producteurs.shape}")
print(f"La taille du fichier ventes est : {ventes.shape}")
print(f"La taille du fichier recoltes est : {recoltes.shape}")

La taille du fichier producteurs est : (31, 8)
La taille du fichier ventes est : (111, 6)
La taille du fichier recoltes est : (91, 5)


### Informations sur chaque fichier

In [4]:
producteurs.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 31 entries, 0 to 30
Data columns (total 8 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   producteur_id         31 non-null     int64  
 1   nom                   31 non-null     object 
 2   prenom                31 non-null     object 
 3   sexe                  31 non-null     object 
 4   region                31 non-null     object 
 5   prefecture            31 non-null     object 
 6   annee_debut_activite  31 non-null     int64  
 7   superficie_ha         29 non-null     float64
dtypes: float64(1), int64(2), object(5)
memory usage: 2.1+ KB


In [5]:
ventes.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 111 entries, 0 to 110
Data columns (total 6 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   vente_id            111 non-null    int64  
 1   recolte_id          111 non-null    int64  
 2   marche              111 non-null    object 
 3   date_vente          111 non-null    object 
 4   prix_unitaire_fcfa  111 non-null    int64  
 5   quantite_vendue_kg  111 non-null    float64
dtypes: float64(1), int64(3), object(2)
memory usage: 5.3+ KB


In [6]:
recoltes.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 91 entries, 0 to 90
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   recolte_id     91 non-null     int64  
 1   producteur_id  91 non-null     int64  
 2   culture        91 non-null     object 
 3   date_recolte   91 non-null     object 
 4   quantite_kg    79 non-null     float64
dtypes: float64(1), int64(2), object(2)
memory usage: 3.7+ KB


### Recherche des valeurs manquantes

In [7]:
print(f"Sur producteurs : {producteurs.isna().sum()}")
print()
print(f"Sur ventes : {ventes.isna().sum()}")
print()
print(f"Sur recoltes : {recoltes.isna().sum()}")

Sur producteurs : producteur_id           0
nom                     0
prenom                  0
sexe                    0
region                  0
prefecture              0
annee_debut_activite    0
superficie_ha           2
dtype: int64

Sur ventes : vente_id              0
recolte_id            0
marche                0
date_vente            0
prix_unitaire_fcfa    0
quantite_vendue_kg    0
dtype: int64

Sur recoltes : recolte_id        0
producteur_id     0
culture           0
date_recolte      0
quantite_kg      12
dtype: int64


### Combien de producteur_id apparaissent dans recoltes mais pas dans producteurs ? Combien de recolte_id apparaissent dans ventes mais pas dans recoltes

In [8]:
prod_invalides = recoltes[~recoltes["producteur_id"].isin(producteurs["producteur_id"])]
print("producteur_id invalides dans recoltes :", prod_invalides["producteur_id"].unique())

producteur_id invalides dans recoltes : [301 305 310]


In [9]:
 recolte_invalides = ventes[~ventes["recolte_id"].isin(recoltes["recolte_id"])]
print("recolte_id invalides dans ventes :", recolte_invalides["recolte_id"].unique())

recolte_id invalides dans ventes : [9002 9003 9001]


# NETTOYAGE

### Suppression des doublons

In [10]:
producteurs = producteurs.drop_duplicates()
ventes = ventes.drop_duplicates()
recoltes = recoltes.drop_duplicates()

In [11]:
print(f"Dupliqué sur producteurs : {producteurs.duplicated().sum()}")
print(f"Dupliqué sur ventes : {ventes.duplicated().sum()}")
print(f"Dupliqué sur recoltes : {recoltes.duplicated().sum()}")

Dupliqué sur producteurs : 0
Dupliqué sur ventes : 0
Dupliqué sur recoltes : 0


### Uniformiser la colonne 'region' de producteurs

In [12]:
producteurs['region'] = producteurs['region'].str.lower().str.title()

### Utilisation de fillna

In [13]:
for df, nom in [(producteurs, "producteurs"), (recoltes, "recoltes"), (ventes, "ventes")]:
    print(nom, df.columns)

producteurs Index(['producteur_id', 'nom', 'prenom', 'sexe', 'region', 'prefecture',
       'annee_debut_activite', 'superficie_ha'],
      dtype='object')
recoltes Index(['recolte_id', 'producteur_id', 'culture', 'date_recolte',
       'quantite_kg'],
      dtype='object')
ventes Index(['vente_id', 'recolte_id', 'marche', 'date_vente', 'prix_unitaire_fcfa',
       'quantite_vendue_kg'],
      dtype='object')


#### Sur superficie_ha

In [14]:
moy_sup_ha = producteurs['superficie_ha'].mean()
producteurs['superficie_ha'] = producteurs['superficie_ha'].fillna(moy_sup_ha)

#### Sur quantite_kg

In [15]:
moy_quant_kg = recoltes['quantite_kg'].mean()
recoltes['quantite_kg']  = recoltes['quantite_kg'].fillna(moy_quant_kg)

# JOINTURE EN CASCADE 

### Jointure interne entre recoltes et producteurs sur 'producteur_id'

- La taille initiale du fichier producteurs est : (31, 8)
- La taille initiale du fichier ventes est : (111, 6)

In [17]:
rp = pd.merge(recoltes, producteurs, on="producteur_id", how="inner")
rp.shape

(81, 12)

### Jointure interne entre ventes et 'rp' sur 'recolte_id'

- La taille initiale du fichier recoltes est : (91, 5)
- La taille du fichier rp est : (81, 12)

In [18]:
full = pd.merge(ventes, rp, on="recolte_id", how="inner")
full.shape

(88, 17)

# COLONNES DERIVEES

In [19]:
full.head()

,vente_id,recolte_id,marche,date_vente,prix_unitaire_fcfa,quantite_vendue_kg,producteur_id,culture,date_recolte,quantite_kg,nom,prenom,sexe,region,prefecture,annee_debut_activite,superficie_ha
0,4002,3008,Marché de Notsé,2024-11-14,150,596.7,9,Coton,2024-02-16,660.3,Sassou,Abla,F,Plateaux,Kpalimé,2006,3.842857
1,4003,3057,Marché d'Adawlato,2023-10-11,150,435.7,11,Maïs,2022-04-16,1045.2,Bissa,Ama,F,Kara,Bassar,2014,4.400000
2,4004,3080,Marché de Kpalimé,2023-11-11,150,934.3,29,Igname,2022-02-13,1713.8,Nyaku,Sena,M,Kara,Kara,2013,1.700000
3,4006,3037,Marché de Dapaong,2024-01-06,250,51.9,8,Riz,2024-06-13,2038.6,Adjovi,Abla,F,Savanes,Dapaong,2009,3.900000
4,4007,3084,Marché de Sokodé,2022-07-18,400,771.1,21,Maïs,2022-07-03,1817.5,Tchamdja,Komla,M,Kara,Kara,2017,3.842857


### Nouvelle colonne

In [26]:
full['revenu_fcfa'] = full['prix_unitaire_fcfa'] * full['quantite_vendue_kg']

### Conversion de la date

In [22]:
full["date_vente"] = pd.to_datetime(full["date_vente"], dayfirst=False, errors="coerce")

In [24]:
full['mois_vente'] = full['date_vente'].dt.month
full['annee_vente'] = full['date_vente'].dt.year

### Nouvelle colonne

In [25]:
def classer_revenu(row):
    if row["revenu_fcfa"] > 300000:
        return "Élevé"
    elif row["revenu_fcfa"] > 100000:
        return "Moyen"
    else:
        return "Faible"
 
full["niveau_revenu"] = full.apply(classer_revenu, axis=1)
print(full["niveau_revenu"].value_counts())

niveau_revenu
Moyen     42
Élevé     25
Faible    21
Name: count, dtype: int64
